# Interactive Simulator — Bloch Sphere

This notebook lets you explore the evolution of a qubit on the Bloch sphere by applying quantum gates interactively, **without needing Streamlit or any external application**.

## How to use it

1. Run all cells (`Kernel → Restart & Run All`).
2. Use the controls in the interactive panel to:
   - Select the initial state of the qubit.
   - Apply fixed gates (H, X, Y, Z, S, T, Sdg, Tdg).
   - Apply parametric gates Rx, Ry, Rz, P with the angle slider.
   - Run predefined sequences.
   - Restart or clear the trajectory.

> **Requirement**: `ipywidgets` must be installed and enabled. If you don't see the controls:
> ```bash
> pip install ipywidgets
> jupyter nbextension enable --py widgetsnbextension
> ```

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.bloch_simulator import BlochSimulator
from src.quantum_math import QuantumMath

print('✓ Modules loaded.')

In [ ]:
# ── Simulator and global state ────────────────────────────────────
sim = BlochSimulator('|0⟩')
gate_history = []


def refresh_display():
    """Updates the figure and the information panel."""
    info = sim.state_info()
    alpha = info['alpha']
    beta  = info['beta']
    bv    = info['bloch_vector']

    # ── Plotly figure ─────────────────────────────────────────────
    fig = sim.plot_trajectory(
        title=f"Bloch sphere  ·  {len(gate_history)} gates applied",
        dark_mode=True,
    )
    fig.update_layout(height=520, width=680)

    with plot_out:
        plot_out.clear_output(wait=True)
        fig.show()

    # ── Information panel ─────────────────────────────────────────
    sign_b = '+' if alpha.imag >= 0 else ''
    sign_c = '+' if beta.imag  >= 0 else ''
    hist_str = ' → '.join(gate_history[-12:]) if gate_history else '(none)'

    info_html.value = f"""
    <div style='font-family:Inter,sans-serif; color:#e6edf3; font-size:13px;
                background:#161b22; border:1px solid #30363d;
                border-radius:10px; padding:14px 18px; line-height:2'>
      <b style='color:#58a6ff'>Current state</b><br>
      <span style='color:#8b949e'>|ψ⟩ =</span>
        <b>{alpha.real:.4f}{sign_b}{alpha.imag:.4f}i</b> |0⟩
        <b> + {beta.real:.4f}{sign_c}{beta.imag:.4f}i</b> |1⟩<br>
      <span style='color:#8b949e'>P(|0⟩) =</span> {info['prob_0']:.4f} &nbsp;
      <span style='color:#8b949e'>P(|1⟩) =</span> {info['prob_1']:.4f}<br>
      <span style='color:#8b949e'>Bloch (x,y,z) =</span>
        ({bv[0]:.3f}, {bv[1]:.3f}, {bv[2]:.3f})<br>
      <span style='color:#8b949e'>θ =</span> {info['theta_deg']:.2f}°&nbsp;
      <span style='color:#8b949e'>φ =</span> {info['phi_deg']:.2f}°<br>
      <br>
      <b style='color:#58a6ff'>History</b><br>
      <span style='color:#f0883e'>{hist_str}</span>
    </div>
    """


print('✓ Visualization functions defined.')

In [ ]:
# ── ipywidgets controls ───────────────────────────────────────────

# Common button style
BTN_STYLE  = dict(button_color='#1f6feb', font_weight='bold')
BTN_LAYOUT = widgets.Layout(width='100%', margin='3px 0')

# ── 1) Initial state ──────────────────────────────────────────────
state_dd = widgets.Dropdown(
    options=list(BlochSimulator.INITIAL_STATES.keys()),
    description='Start:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='100%'),
)
btn_reset = widgets.Button(
    description='🔄 Restart',
    style=BTN_STYLE, layout=BTN_LAYOUT,
)

# ── 2) Fixed gates ────────────────────────────────────────────────
gate_dd = widgets.Dropdown(
    options=['H', 'X', 'Y', 'Z', 'S', 'T', 'Sdg', 'Tdg', 'I'],
    description='Gate:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='100%'),
)
btn_gate = widgets.Button(
    description='Apply fixed gate',
    style=BTN_STYLE, layout=BTN_LAYOUT,
)

# ── 3) Parametric gates ───────────────────────────────────────────
param_dd = widgets.Dropdown(
    options=['Rx', 'Ry', 'Rz', 'P'],
    description='Gate:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='100%'),
)
angle_slider = widgets.IntSlider(
    value=90, min=-360, max=360, step=5,
    description='Angle (°):',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='100%'),
)
btn_param = widgets.Button(
    description='Apply parametric gate',
    style=BTN_STYLE, layout=BTN_LAYOUT,
)

# ── 4) Predefined sequences ───────────────────────────────────────
SEQUENCES = {
    'Simple Hadamard (H)': [{'gate': 'H'}],
    'Superposition + phase (H → S)': [{'gate': 'H'}, {'gate': 'S'}],
    'Teleportation (H → Z → H)': [
        {'gate': 'H'}, {'gate': 'Z'}, {'gate': 'H'}],
    'Full X rotation (4×Rx45°)': [
        {'gate': 'Rx', 'theta': np.pi/4}] * 4,
    'T⁸ = I': [{'gate': 'T'}] * 8,
    'Bloch tour (H → S → T → H)': [
        {'gate': 'H'}, {'gate': 'S'}, {'gate': 'T'}, {'gate': 'H'}],
}
seq_dd = widgets.Dropdown(
    options=list(SEQUENCES.keys()),
    description='Sequence:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='100%'),
)
btn_seq = widgets.Button(
    description='▶ Apply sequence',
    style=BTN_STYLE, layout=BTN_LAYOUT,
)

# ── 5) Clear trajectory ───────────────────────────────────────────
btn_clear = widgets.Button(
    description='🗑️ Clear trajectory',
    style=dict(button_color='#6e7681', font_weight='bold'),
    layout=BTN_LAYOUT,
)

# ── Outputs ───────────────────────────────────────────────────────
plot_out  = widgets.Output()
info_html = widgets.HTML()


# ── Callbacks ─────────────────────────────────────────────────────
def on_reset(_):
    global gate_history
    sim.set_initial_state(state_dd.value)
    gate_history = []
    refresh_display()

def on_gate(_):
    g = gate_dd.value
    sim.apply_gate(g)
    gate_history.append(g)
    refresh_display()

def on_param(_):
    g   = param_dd.value
    ang = np.radians(angle_slider.value)
    label_ang = f'{angle_slider.value}°'
    if g == 'P':
        sim.apply_gate(g, phi=ang)
    else:
        sim.apply_gate(g, theta=ang)
    gate_history.append(f'{g}({label_ang})')
    refresh_display()

def on_seq(_):
    for step in SEQUENCES[seq_dd.value]:
        g = step['gate']
        kw = {k: v for k, v in step.items() if k != 'gate'}
        sim.apply_gate(g, **kw)
        gate_history.append(g)
    refresh_display()

def on_clear(_):
    current = sim.state
    sim.set_initial_state('|0⟩')
    sim._state = current
    bv = QuantumMath.bloch_vector(current)
    sim._trajectory = [bv]
    sim._gate_labels = []
    gate_history.clear()
    refresh_display()

btn_reset.on_click(on_reset)
btn_gate.on_click(on_gate)
btn_param.on_click(on_param)
btn_seq.on_click(on_seq)
btn_clear.on_click(on_clear)


# ── Control panels ────────────────────────────────────────────────
def section(title, *children):
    header = widgets.HTML(
        f"<b style='color:#58a6ff;font-family:Inter,sans-serif;'"
        f">{title}</b>"
    )
    return widgets.VBox([header, *children],
                        layout=widgets.Layout(
                            border='1px solid #30363d', border_radius='8px',
                            padding='10px', margin='6px 0',
                            background_color='#161b22'))

sidebar = widgets.VBox([
    section('Initial state', state_dd, btn_reset),
    section('Fixed gate', gate_dd, btn_gate),
    section('Parametric gate', param_dd, angle_slider, btn_param),
    section('Sequences', seq_dd, btn_seq),
    btn_clear,
], layout=widgets.Layout(width='300px', padding='6px'))

right_panel = widgets.VBox(
    [plot_out, info_html],
    layout=widgets.Layout(flex='1', padding='6px'),
)

ui = widgets.HBox(
    [sidebar, right_panel],
    layout=widgets.Layout(
        background_color='#0d1117',
        border='1px solid #21262d',
        border_radius='12px',
    ),
)

# First render
display(ui)
refresh_display()

## Educational section — Quick reference

### Basic states

| State | α | β | Position on Bloch |
|-------|---|---|-------------------|
| `\|0⟩` | 1 | 0 | North pole (0,0,1) |
| `\|1⟩` | 0 | 1 | South pole (0,0,−1) |
| `\|+⟩` | 1/√2 | 1/√2 | Equator (+x) |
| `\|-⟩` | 1/√2 | −1/√2 | Equator (−x) |
| `\|i⟩` | 1/√2 | i/√2 | Equator (+y) |
| `\|−i⟩` | 1/√2 | −i/√2 | Equator (−y) |

### 1-qubit gates

| Gate | Effect on Bloch | Matrix |
|------|-----------------|--------|
| H | Hadamard: swaps X↔Z | [[1,1],[1,−1]]/√2 |
| X | π rotation on X (NOT) | [[0,1],[1,0]] |
| Y | π rotation on Y | [[0,−i],[i,0]] |
| Z | π rotation on Z (phase flip) | [[1,0],[0,−1]] |
| S | π/2 rotation on Z | [[1,0],[0,i]] |
| T | π/4 rotation on Z | [[1,0],[0,e^{iπ/4}]] |
| Rx(θ) | θ radian rotation on X | cos(θ/2)I − i sin(θ/2)X |
| Ry(θ) | θ radian rotation on Y | cos(θ/2)I − i sin(θ/2)Y |
| Rz(θ) | θ radian rotation on Z | cos(θ/2)I − i sin(θ/2)Z |

### Bloch vector formulas

The general state of a pure qubit is parameterized as:

$$|\psi\rangle = \cos\!\left(\frac{\theta}{2}\right)|0\rangle + e^{i\varphi}\sin\!\left(\frac{\theta}{2}\right)|1\rangle$$

with the Bloch vector:

$$\vec{r} = (\sin\theta\cos\varphi,\; \sin\theta\sin\varphi,\; \cos\theta)$$

In [ ]:
# ── Free exploration: custom experiment ───────────────────────────
# Modify the following lines and run the cell.

exp = BlochSimulator('|0⟩')

exp.apply_sequence([
    {'gate': 'H'},
    {'gate': 'Rx', 'theta': np.pi / 3},
    {'gate': 'S'},
    {'gate': 'T'},
    {'gate': 'H'},
])

info = exp.state_info()
print(f"Final state:")
print(f"  α = {info['alpha']:.4f}")
print(f"  β = {info['beta']:.4f}")
print(f"  P(|0⟩) = {info['prob_0']:.4f}")
print(f"  Bloch  = {tuple(round(c, 3) for c in info['bloch_vector'])}")

fig_exp = exp.plot_trajectory(title='Custom experiment', dark_mode=True)
fig_exp.update_layout(height=480, width=700)
fig_exp.show()